In [ ]:
"""
Railway budget problem - QAOA pe nucleul de sine (max 15 qubiti IN TOTAL).

Pipeline hibrid:
  1) CLASIC : greedy dupa profit/cost -> punct de rupere -> alegem un nucleu de sine.
              Sinele clar bune (inainte) = fixate pe 1, cele clar proaste (dupa) = pe 0.
  2) CUANTIC: nucleul + variabile slack devin qubiti. Constrangerea de buget intra in
              Hamiltonian ca penalizare patratica => termeni ZZ (problema devine cuplata).
              Circuit QAOA cu p straturi: |+>^n -> [exp(-i*gamma*H_C) -> RX(2*beta)] x p
  3) CLASIC : un optimizator (COBYLA) ajusteaza gamma/beta; masuram circuitul (shots),
              iar din bitstring-urile obtinute o pastram pe cea mai buna FEZABILA.

Circuitul e simulat prin statevector in numpy (adica exact ce face un simulator Qiskit),
deci ruleaza fara Qiskit. Lista Pauli de la `hamiltonian_pauli` e in conventia Qiskit
(qubitul 0 = caracterul din dreapta) si poate fi data direct lui SparsePauliOp.from_list.
"""
import time
import numpy as np
from scipy.optimize import minimize


# ==========================================
# 0. ETALON EXACT
# ==========================================
def knapsack_dp(profituri, costuri, buget):
    n = len(profituri)
    dp = [0] * (buget + 1)
    pastrat = [[False] * (buget + 1) for _ in range(n)]
    for i in range(n):
        for b in range(buget, costuri[i] - 1, -1):
            if dp[b - costuri[i]] + profituri[i] > dp[b]:
                dp[b] = dp[b - costuri[i]] + profituri[i]
                pastrat[i][b] = True
    b, alese = buget, []
    for i in range(n - 1, -1, -1):
        if pastrat[i][b]:
            alese.append(i)
            b -= costuri[i]
    return dp[buget], sorted(alese)


# ==========================================
# 1. PARTEA CLASICA: nucleul
# ==========================================
def coeficienti_slack(buget_ramas, m):
    """Variabile slack care reprezinta EXACT valorile 0..buget_ramas (suma maxima = buget_ramas)."""
    if m == 0:
        return []
    return [2 ** j for j in range(m - 1)] + [buget_ramas - (2 ** (m - 1) - 1)]


def alege_nucleu(profituri, costuri, buget, max_qubiti=15):
    """Nucleu + slack incap impreuna in `max_qubiti` qubiti."""
    n = len(profituri)
    ordine = sorted(range(n), key=lambda i: profituri[i] / costuri[i], reverse=True)
    cost, rupere = 0, n
    for poz, i in enumerate(ordine):
        if cost + costuri[i] > buget:
            rupere = poz
            break
        cost += costuri[i]
    for k in range(max_qubiti, 0, -1):
        inc = max(0, rupere - k // 2)
        sf = min(n, inc + k)
        inc = max(0, sf - k)
        fixate_1, nucleu = ordine[:inc], ordine[inc:sf]
        buget_ramas = buget - sum(costuri[i] for i in fixate_1)
        m = buget_ramas.bit_length() if buget_ramas > 0 else 0
        if len(nucleu) + m <= max_qubiti:
            return fixate_1, nucleu, buget_ramas, m
    raise ValueError("Nu incape niciun nucleu in numarul de qubiti ceruti.")


# ==========================================
# 2. HAMILTONIANUL (Ising, cu termeni ZZ)
# ==========================================
def hamiltonian_pauli(profituri_n, ponderi, buget_ramas, A):
    """
    H = -sum_i p_i x_i  +  A * (sum_j w_j x_j - B)^2,     x = (1 - Z)/2
    `ponderi` = costurile sinelor din nucleu + coeficientii slack (N valori).
    Intoarce lista [(sir_pauli, coeficient)] in conventia Qiskit.
    """
    N = len(ponderi)
    p = list(profituri_n) + [0] * (N - len(profituri_n))
    w = ponderi
    a = [-p[j] + A * (w[j] ** 2 - 2 * buget_ramas * w[j]) for j in range(N)]  # liniar in x
    q = {(j, l): 2 * A * w[j] * w[l] for j in range(N) for l in range(j + 1, N)}  # patratic

    def sir(*qubiti):
        s = ["I"] * N
        for j in qubiti:
            s[N - 1 - j] = "Z"  # Qiskit: qubitul 0 este caracterul din dreapta
        return "".join(s)

    identitate = A * buget_ramas ** 2 + sum(a) / 2 + sum(q.values()) / 4
    termeni = [("I" * N, identitate)]
    for j in range(N):
        coef = -a[j] / 2 - sum(v for (x, y), v in q.items() if j in (x, y)) / 4
        termeni.append((sir(j), coef))
    for (j, l), v in q.items():
        termeni.append((sir(j, l), v / 4))
    return termeni


def energii_din_pauli(termeni, N):
    """Diagonala lui H pe toate cele 2^N stari de baza, calculata DIN lista Pauli."""
    x = np.arange(2 ** N, dtype=np.int64)
    z = 1 - 2 * ((x[:, None] >> np.arange(N)) & 1)  # z[:, q] = +1 daca qubitul q e 0, -1 daca e 1
    E = np.zeros(2 ** N)
    for sir, coef in termeni:
        v = np.full(2 ** N, coef)
        for pos, ch in enumerate(sir):
            if ch == "Z":
                v = v * z[:, N - 1 - pos]
        E += v
    return E


# ==========================================
# 3. CIRCUITUL QAOA (statevector)
# ==========================================
def mixer_rx(psi, beta, N):
    """RX(2*beta) pe fiecare qubit: cos(beta)*I - i*sin(beta)*X."""
    c, s = np.cos(beta), -1j * np.sin(beta)
    for q in range(N):
        v = psi.reshape(2 ** (N - 1 - q), 2, 2 ** q)
        psi = (c * v + s * v[:, ::-1, :]).reshape(-1)
    return psi


def stare_qaoa(params, E, N, p):
    gamma, beta = params[:p], params[p:]
    psi = np.full(2 ** N, 2 ** (-N / 2), dtype=complex)  # H pe toti qubitii -> |+>^N
    for l in range(p):
        psi = psi * np.exp(-1j * gamma[l] * E)  # stratul de cost  exp(-i*gamma*H_C)
        psi = mixer_rx(psi, beta[l], N)         # stratul de mixer exp(-i*beta*sum X)
    return psi


def rezolva_qaoa(profituri, costuri, buget, p=3, restarturi=4, max_iter=200,
                 shots=1024, max_qubiti=15, seed=0, verbose=True):
    rng = np.random.default_rng(seed)
    t0 = time.time()

    fixate_1, nucleu, B_rez, m = alege_nucleu(profituri, costuri, buget, max_qubiti)
    Pn = [profituri[i] for i in nucleu]
    Cn = [costuri[i] for i in nucleu]
    k, N = len(nucleu), len(nucleu) + m
    ponderi = Cn + coeficienti_slack(B_rez, m)

    A = max(Pn) + 1  # penalizare mai mare decat castigul maxim al unei singure sine
    termeni = hamiltonian_pauli(Pn, ponderi, B_rez, A)
    E = energii_din_pauli(termeni, N)

    # verificare: lista Pauli reproduce exact functia QUBO
    x = np.arange(2 ** N, dtype=np.int64)
    biti = (x[:, None] >> np.arange(N)) & 1
    E_qubo = -(biti[:, :k] @ np.array(Pn)) + A * (biti @ np.array(ponderi) - B_rez) ** 2
    assert np.allclose(E, E_qubo), "Hamiltonianul Pauli nu coincide cu QUBO!"

    En = (E - E.min()) / (E.max() - E.min()) * 10  # doar rescalare (aceeasi stare fundamentala)

    def energie_medie(params):
        psi = stare_qaoa(params, En, N, p)
        return float(np.real(np.sum(np.abs(psi) ** 2 * En)))

    # --- bucla hibrida: QPU (circuitul) + optimizator clasic ---
    best, evaluari = None, 0
    for r in range(restarturi):
        t = (np.arange(p) + 0.5) / p
        if r == 0:
            x0 = np.concatenate([0.4 * t, 0.4 * (1 - t)])  # rampa liniara (tip recoacere)
        else:
            x0 = rng.uniform(0, 0.8, 2 * p)
        res = minimize(energie_medie, x0, method="COBYLA", options={"maxiter": max_iter, "rhobeg": 0.1})
        evaluari += res.nfev
        if best is None or res.fun < best.fun:
            best = res

    # --- masurare: shots pe circuitul optimizat ---
    psi = stare_qaoa(best.x, En, N, p)
    prob = np.abs(psi) ** 2
    prob /= prob.sum()
    probe = rng.choice(2 ** N, size=shots, p=prob)
    uniform = rng.choice(2 ** N, size=shots)  # etalon: fara circuit, stari la intamplare

    def cea_mai_buna_fezabila(esantioane):
        best_p, best_b = -1, None
        for s in np.unique(esantioane):
            b = biti[s, :k]
            if int(b @ np.array(Cn)) <= B_rez and int(b @ np.array(Pn)) > best_p:
                best_p, best_b = int(b @ np.array(Pn)), b
        return best_p, best_b

    pq, bq = cea_mai_buna_fezabila(probe)
    pu, _ = cea_mai_buna_fezabila(uniform)
    profit_fix = sum(profituri[i] for i in fixate_1)

    stari_optime = E == E.min()
    rez = {
        "profit": profit_fix + pq if bq is not None else None,
        "profit_uniform": profit_fix + pu if pu >= 0 else None,
        "sine": sorted(fixate_1 + [nucleu[j] for j in range(k) if bq is not None and bq[j]]),
        "qubiti": N, "sine_in_nucleu": k, "qubiti_slack": m, "p": p,
        "termeni_ZZ": sum(1 for s, _ in termeni if s.count("Z") == 2),
        "evaluari_circuit": evaluari,
        "prob_pe_optim": float(prob[stari_optime].sum()),
        "prob_uniform_pe_optim": float(stari_optime.sum() / 2 ** N),
        "timp_s": time.time() - t0,
    }
    if verbose:
        print(f"Circuit: {N} qubiti ({k} sine + {m} slack), {rez['termeni_ZZ']} termeni ZZ, p={p}, "
              f"{evaluari} evaluari de circuit, {shots} shots, {rez['timp_s']:.1f}s")
        print(f"Probabilitatea starii optime: QAOA {rez['prob_pe_optim']:.4f} "
              f"vs uniform {rez['prob_uniform_pe_optim']:.6f} "
              f"(x{rez['prob_pe_optim'] / rez['prob_uniform_pe_optim']:.1f})")
    return rez


# ==========================================
# 4. EXECUTIE (aceeasi instanta ca in codul tau)
# ==========================================
if __name__ == "__main__":
    np.random.seed(42)
    NR_SINE = 150
    profituri = [int(x) for x in np.random.randint(2, 15, NR_SINE)]
    costuri = [int(x) for x in np.random.randint(1, 8, NR_SINE)]
    BUGET = 40

    opt, _ = knapsack_dp(profituri, costuri, BUGET)
    print(f"Optim exact (DP): {opt}\n")

    r = rezolva_qaoa(profituri, costuri, BUGET)
    cost = sum(costuri[i] for i in r["sine"])
    print(f"\nProfit QAOA           : {r['profit']}  (cost {cost} / buget {BUGET})")
    print(f"Profit esantionare random (aceleasi shots): {r['profit_uniform']}")
    assert cost <= BUGET
    print(f"Sine selectate        : {r['sine']}")
    print("Coincide cu optimul   :", r["profit"] == opt)x``


Optim exact (DP)   : profit 282, cost 40
Metoda cu nucleu   : profit 282, cost 40 (buget 40)
Șine selectate     : [2, 3, 5, 11, 40, 44, 55, 58, 61, 63, 66, 69, 74, 77, 79, 82, 84, 89, 95, 98, 99, 101, 102, 103, 109, 132, 133]
Coincide cu optimul: True
